# 04 — Train Classifier
Supervised multi-class classification (normal + 4 attack categories).

In [ ]:
import sys
sys.path.insert(0, '..')

import pickle
import torch

from src.config import (
    DEVICE, CLS_HIDDEN_DIMS, CLS_LR, CLS_EPOCHS, CLS_PATIENCE,
    CLASSIFIER_CHECKPOINT, PROCESSED_DIR, BATCH_SIZE, NUM_CLASSES
)
from src.models import Classifier
from src.dataset import make_loader
from src.train_utils import train_classifier
from src.visualize import plot_loss_curves, plot_accuracy_curves

print(f'Device: {DEVICE}')

## Load data

In [ ]:
with open(PROCESSED_DIR / 'data.pkl', 'rb') as f:
    data = pickle.load(f)

input_dim = data['X_train'].shape[1]
print(f'Train samples : {len(data["X_train"]):,}')
print(f'Val samples   : {len(data["X_val"]):,}')
print(f'Input dim     : {input_dim}')
print(f'Num classes   : {NUM_CLASSES}')

## Build model & loaders

In [ ]:
model = Classifier(input_dim=input_dim, hidden_dims=CLS_HIDDEN_DIMS, num_classes=NUM_CLASSES)
model = model.to(DEVICE)
print(model)

optimizer = torch.optim.Adam(model.parameters(), lr=CLS_LR)

train_loader = make_loader(data['X_train'], data['y_cls_train'], shuffle=True, batch_size=BATCH_SIZE)
val_loader   = make_loader(data['X_val'],   data['y_cls_val'],   shuffle=False, batch_size=BATCH_SIZE)

## Train

In [ ]:
history = train_classifier(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=DEVICE,
    epochs=CLS_EPOCHS,
    patience=CLS_PATIENCE,
    checkpoint_path=CLASSIFIER_CHECKPOINT,
)
print(f'\nBest checkpoint: {CLASSIFIER_CHECKPOINT}')

## Training curves

In [ ]:
plot_loss_curves(history, title='Classifier Loss Curves', filename='cls_loss_curves.png')
plot_accuracy_curves(history, filename='cls_accuracy_curves.png')